# 01 — EDA

Τι απαντάει αυτό το notebook, με τη σειρά:

1. **Διπλότυπα** — πόσα και τι είδους. Ορίζει αν χρειάζεται group split.
2. **`features`** — σπάει σε στήλες ή μένει ενιαίο κείμενο;
3. **Συνένωση πεδίων** — ένα text field ή ξεχωριστό TF-IDF ανά πεδίο;
4. **`brand`** — αξίζει ως ξεχωριστό feature;
5. **Τελικές αποφάσεις** — πίνακας με ό,τι πάει στο training.

Κάθε ενότητα κλείνει με «Συμπέρασμα» που φέρει το νούμερο στο οποίο στηρίζεται.

In [2]:
import pandas as pd
from skrub import TableReport

TARGET_COLUMN = "category"
df = pd.read_csv("../data/raw/products.csv")

In [11]:
# TableReport(df).open()
df["category"].value_counts().sort_values(ascending=False)

category
5.0       70062
1049.0    20088
1313.0    10313
0.0        2978
1.0        1812
1053.0      969
9.0         706
3.0         333
1842.0      311
7.0         269
Name: count, dtype: int64

In [4]:
rows_before = len(df)
df = df.drop_duplicates()
print(f"dropped {rows_before - len(df):,}")
TableReport(df).open()


σβήστηκαν 6,637


Processing column   5 / 5


## 2. FEATURES 

In [5]:
"""Eyeballing the ';' separator: if the counts vary wildly between rows,
splitting into columns is not worth it, we would just create empty cells."""


semicolons_per_product = df["features"].str.count(";")
semicolons_per_product.value_counts().sort_index().rename_axis("n_semicolons").reset_index(name="n_rows")



,n_semicolons,n_rows
0,0.0,1263
1,1.0,1681
2,2.0,5501
3,3.0,10060
4,4.0,29494
...,...,...
82,96.0,1
83,100.0,1
84,111.0,1
85,116.0,1


### Συμπέρασμα Δεν σπαω εχουν υπερβολικα ανισο πληθος

## 3. ΣΥΝΕΝΩΣΗ ΠΕΔΙΩΝ?



                                        Cramer V apo Table reports

product_description	        brand	    0.429	 =>Συσχετιση(product_description,brand) μετρια-προς δυνατη

product_description	        features	0.346
=>Συσχετιση(product_description, features) μετρια


In [6]:
description_and_features = df.dropna(subset=["product_description", "features"])
duplicate_pairs = description_and_features.groupby(["product_description", "features"]).size()

print(f"distinct text pairs = {len(duplicate_pairs)}")
print(f"pairs with 2+ copies = {(duplicate_pairs > 1).sum()}")
print(f"largest number of copies = {duplicate_pairs.max()}")


διαφορετικά ζεύγη κειμένων= 34845
ζεύγη με 2+ αντίγραφα= 7961
μέγιστα αντίγραφα ζεύγους= 479


Η συσχετιση που υπεθεσε η κραμερ τιμη, ηταν πιθανον εξαιτιας πολλων αντιγραφων.
Risk gia data leakage. 

- Αν κανω διαγραφη αντιγραφων δεν εχω οβερφιτ
στην εκπαιδευση αλλα διαφερει απο την παραγωγη.Mετράει 1 λάθος,ενώ είναι Ν λάθος προϊόντα ιδια μεταξυ τους. 
παραπλανουςν τα μετρικς που θα βγουν κ παραπλανουν ολο το pipeline

- Αν διαγραφή μόνο στο συνολο εκπαιδευσης αλλα στο συνολο τεστ, μεινουν τα αντίγραφα. 
 Αυτο ειναι σωστο για πελάτη. 

In [7]:
missing_description_by_category = df["product_description"]\
    .isna()\
    .groupby(df[TARGET_COLUMN])\
    .mean()\
    .mul(100)\
    .round(1)\
    .sort_values(ascending=False)\
    .rename(f"perc(%) kena description ana {TARGET_COLUMN}")

missing_description_by_category

display(missing_description_by_category.sort_values(ascending=False).round(3))

brand_counts = df["brand"].value_counts()
print(f"brands: {brand_counts.size:,} | holding a single product: {(brand_counts == 1).mean():.1%}")

category
1842.0    60.5
1053.0    43.2
1049.0    37.7
0.0       37.2
5.0       36.8
3.0       18.3
7.0       16.4
1313.0    12.7
9.0        8.0
1.0        7.8
Name: perc(%) kena description ana category, dtype: float64

brands: 9,570 | με 1 μόνο προϊόν: 50.5%


In [8]:
# Many empties, but no extra flag: in production we cannot know whether they
# will keep arriving empty. Just fill for TF-IDF.
df["product_description"].fillna("")



0         Pete the Cat is the coolest ; most popular cat...
1         The New Yorker Handsome Cello Wrapped Hard Mag...
2                                                          
3         Men'S Full Sleeve Raglan T-Shirts Denim T-Shir...
4         Wild Animals are the animals that mostly stays...
                                ...                        
107836    We brings to you comfortable t-shirts in attra...
107837    eCools Polo T-shirts are some of the most vers...
107838    <p><strong>Brand Name:</strong> NIGHTY HOUSE</...
107839    KIPA Tees are available in various fabrics ; j...
107840    ILLI LONDON clothing is stop destination that ...
Name: product_description, Length: 101204, dtype: str

In [9]:
def vocab_overlap(column_a: str, column_b: str) -> float:
    """Jaccard vocabulary overlap between two fields. High overlap means same nature, so joining them would be reasonable.

    """
    vocabulary_a = set(df[column_a].dropna().str.lower().str.split().explode())
    vocabulary_b = set(df[column_b].dropna().str.lower().str.split().explode())

    intersection = vocabulary_a.intersection(vocabulary_b)
    union = vocabulary_a.union(vocabulary_b)

    return len(intersection) / len(union)


for column_a, column_b in [
    ("product_name", "product_description"),
    ("product_name", "features"),
    ("product_description", "features"),
    ]:
    print(f"{column_a} vs {column_b}: {vocab_overlap(column_a, column_b):.3f}")
    
    vocabulary_a = set(df[column_a].dropna().str.lower().str.split().explode())
    vocabulary_b = set(df[column_b].dropna().str.lower().str.split().explode())

    containment = len(vocabulary_a & vocabulary_b) / min(len(vocabulary_a), len(vocabulary_b))
    print(containment)


product_name vs product_description: 0.110
0.22045574321780811
product_name vs features: 0.109
0.21163775599002146
product_description vs features: 0.201
0.4051894181122005


### Συμπέρασμα 3 — τα πεδία ΔΕΝ ενώνονται σε ένα text field

| Ζεύγος | Jaccard | Containment |
|---|---|---|
| name και description | 0,110 | 0,220 |
| name και features | 0,109 | 0,212 |
| description και features | 0,201 | 0,405 |

Μικρά νούμερα γενικά: ακόμη και το πιο συγγενικό ζεύγος
(description με features) μοιράζεται μόλις το 41% του μικρότερου
λεξιλογίου. Κάθε πεδίο έχει δικές του λέξεις που αξίζει να τις δει
ξεχωριστά το μοντέλο.

**Απόφαση: ξεχωριστό TF-IDF ανά πεδίο.** Το containment στηρίζει την
απόφαση, αλλά οι κύριοι λόγοι είναι δύο:
1. διατηρείται η **προέλευση** της λέξης — "cotton" στο `product_name`
   δεν είναι το ίδιο σήμα με "cotton" θαμμένο στην 5η γραμμή των features·
2. το μακρύ `description` (~50 λέξεις) δεν **πνίγει** το σύντομο
   `product_name` (~12 λέξεις), που τελικά αποδείχθηκε το πιο σημαντικό
   πεδίο (49,2% της σημαντικότητας — βλ. `docs/experiment-01-v1.md` §Γ5).

Το ενωμένο text field κρατιέται ως baseline σύγκρισης (experiment 03).

### Brand purity

Ανά brand se πόσες κατηγορίες εμφανίζεται.
Purity κοντά στο 1 σημαίνει ότι το brand είναι μόνο του ισχυρός προβλέπτης, άρα μπαίνει ως ξεχωριστό categorical και όχι μέσα στο TF-IDF.

In [ ]:
min_count=20
brand_category = pd.crosstab(df["brand"], df[TARGET_COLUMN], normalize="index")
brand_purity = brand_category.max(axis=1)

brand_counts = df["brand"].value_counts()
frequent_brands = brand_counts[brand_counts >= min_count].index
print("frequent_brands")
print(f"mean purity brand: {brand_purity[frequent_brands].mean():.1%}")



mean purity brand: 90.4%


### Συμπέρασμα — το brand είναι ισχυρός προβλέπτης, αλλά υψηλής πληθικότητας

Μέση purity **90,4%** στα 735 brands με ≥20 προϊόντα (χωρίς στάθμιση·
σταθμισμένη ανά προϊόν πέφτει στο 85,2%), έναντι baseline **65%** που είναι
το μερίδιο της κυρίαρχης κλάσης. Δηλαδή: ξέροντας μόνο το brand, μαντεύεις
σωστά πολύ πιο συχνά από το να λες πάντα "κλάση 5".

Όμως **9.570** μοναδικά brands με το **50,5%** να έχει ένα μόνο προϊόν —
one-hot θα έδινε 9.570 στήλες θορύβου.

**Απόφαση:** `OrdinalEncoder(min_frequency=20)` και δήλωση ως native
categorical στο LightGBM· τα σπάνια brands πέφτουν όλα σε μία κοινή
κατηγορία αντί να γίνουν το καθένα δική του στήλη.

**Αντίφαση με αυτόματο εργαλείο:** το skrub έδωσε Cramér's V = 0,0814 για
brand με category, δηλαδή "σχεδόν καμία σχέση". Κρατήθηκε η απευθείας
μέτρηση (purity), γιατί το V φουσκώνει ή καταρρέει ανάλογα με το πώς
ομαδοποιούνται 9.570 τιμές.

In [15]:
from pathlib import Path

processed_dir = Path("../proc/processed")
df.to_csv("../data/proc/proc_products.csv", index=False)


## 4. Τελικές αποφάσεις — τι πάει στο training

| # | Ερώτημα | Απόφαση | Τεκμήριο |
|---|---|---|---|
| 1 | Σπάει το `features` σε στήλες; | **ΟΧΙ** — ενιαίο κείμενο | 0–120 `;` ανά προϊόν, 18,17% bullets με πεζό αρχικό |
| 2 | Χρειάζεται καθάρισμα του κειμένου; | **ΟΧΙ** — μόνο `fillna("")` | ο `TfidfVectorizer` πετάει μόνος του `[ ] ;` |
| 3 | Ενωμένο text field; | **ΟΧΙ** — 3 ξεχωριστά TF-IDF | containment 0,21–0,41 |
| 4 | brand ξεχωριστά; | **ΝΑΙ** — `OrdinalEncoder(min_frequency=20)`, native categorical | purity 90,4% vs baseline 65%, 9.570 brands |
| 5 | `has_description` flag; | **ΟΧΙ** | σήμα υπάρχει (7,8%–60,5%), αλλά οι 3 μεγαλύτερες κλάσεις είναι όλες ~37% οπότε υπάρχει ρίσκο data drift χωρίς κέρδος |
| 6 | Διπλότυπα; | **πλήρη αντίγραφα σβήνονται** (από 107.841 μένουν 101.204 γραμμές)· οι παραλλαγές κειμένου **μένουν** | οι παραλλαγές είναι Size S / Size M, υπάρχουν και στην παραγωγή |
| 7 | Πώς γίνεται το split; | **group split** ανά `(product_description, features)` | 7.961 ζεύγη με 2+ αντίγραφα, το μεγαλύτερο ×479 αλλιώς προκύπτει leakage |
| 8 | Μετρική; | **macro F1** + `class_weight="balanced"` | οδηγία πελάτη: όλες οι κλάσεις ίσης σημασίας |

Το αρχείο που παράγεται εδώ (`../data/proc/proc_products.csv`) είναι το
dedup-αρισμένο dataset. Το `02_training.ipynb` ξαναδιαβάζει το raw και
επαναλαμβάνει το `drop_duplicates()`, ώστε το training να μην εξαρτάται από
την εκτέλεση του EDA.

**Μένει ανοιχτό για το training:** το brand έδειξε purity 90,4% εδώ, αλλά
αυτό δεν σημαίνει ότι προσθέτει **επιπλέον** πληροφορία — το brand
γράφεται ήδη μέσα στο `product_name`. Ελέγχεται με feature importance
(§Γ5) και με ablation (experiment 03).